# **MODELO DIMENSIONAL PARA EL SISTEMA DE COMPRAS** 

1. **PROCESO DEL NEGOCIO**
El cliente nos informa de que lo que busca es analizar a sus proveedores. Quiere saber a qué proveedor le compra más y cuántos productos diferentes le compra a dicho proveedor.
También nos comenta que quiere analizar la calidad del proveedor a partir de los tiempos de entrega del mismo.  
Para analizar esto, vamos a definir dos procesos de negocio:
- <b>Importancia del proveedor</b>:
    - Importe total comprado
    - Número de productos diferentes comprados
- <b>Calidad del proveedor</b>:
    - Retraso medio enttre lead time real y teórico (teniendo en cuenta el proveedor en general, pero también por produto)

2. **GRANULARIDAD**
Tenemos que determinar ahora el nivel mínimo de detalle. Por una parte, nos comenta que pra la importancia del proveedor quiere disponer de un detalle diario, aunque se verá normalmente a nivel de detalle mensual. 
En cuanto a la calidad, nos comenta que se elegirá un periodo de tiempo largo, como un año, para no distorsionar la media.
Pero la importancia no es solo a nivel de proveedor, sino que interesa a nivel de producto de cada proveedor.

Tenemos que definir el nivel más bajo entonces por <b>línea de factura</b>. Donde veremos qué producto se ha pedido, a qué proveedor, importe de dicho producto, fechas de pedido y de entrega.

Tendremos que definir también agregaciones diarias, con capacidad para agregar en mensuales y anuales.

3. **DIMENSIONES**
Nos quedamos finalmente con tres tablas que nos serivarán para este análisis. 
- **Products**: en esta tabla se detallaran el ID del producto y su descipción, para poder identificar fácilmente qué producto se está comprando
- **Invoice_Products**: donde tendremos qué producto se ha comprado, en qué cantidad, qué importe
- **Header + Supplier**: donde se podrá hacer un análisis de los lead times de los proveedores, comparando el lead time real con el teórico
- **Tiempo**

4. **TABLA DE HECHOS**

5. **MEDIDAS CLAVES**
- Total de producto comprado a un proveedor
- Total de productos diferentes comrpados al mismo proveedor
- Tiempo promedio de cada proveedor

6. **DIAGRAMA**
Lo representamos en un diagrama estrella.

                    [Dimensión proveedor]
                            |
[Dimensión tiempo] — [Hecho compra] — [Dimensión producto]
                            |
                    [Dimensión tiempo]



In [1]:
# cargamos librerias
import pandas as pd
import numpy as np

In [2]:
# cargamos los datos
df_header = pd.read_csv("invoices_header.csv", delimiter= ";")
# invoice; inbound date; supplier; order date; invoice date
df_invoices = pd.read_csv("invoices_products.csv", delimiter= ";")
# invoice; quantity; product; purchase price (unit); section
df_supplier = pd.read_csv("suppliers.csv", delimiter = ";")
# product; type; division; group; short text; description
df_product = pd.read_csv("products.csv", delimiter = ";")
# supplier name; payment method; payment terms; ID supplier; country; currency

In [3]:
# vamos a ir analizando modificando poco a poco los df que tenemos

### DF_HEADER ###


df_header.head()

,Invoice,InboundDate,Supplier,OrderDate,InvoiceDate
0,FFCC141196,2014-05-26,PROV1650,2014-05-10,2014-05-29
1,FFCC141197,2014-05-20,PROV40000235,2014-04-13,2014-05-23
2,FFCC141198,2014-05-12,PROV1647,2014-04-27,2014-05-15
3,FFCC141199,2014-05-19,PROV40001000,2014-03-24,2014-05-22
4,FFCC141200,2014-05-24,PROV40000850,2014-05-11,2014-05-27


In [4]:
# vamos a empezar mirando si tenemos valores nulos
# es importante no tenerlos en invoice, que es nuestra key
# no hay valores nulos, así que seguimos
for column in df_header.columns:
    null_values_header = df_header[column].isnull().sum()
    print(f"Null values in '{column}':")
    print(null_values_header)
    print("---")

Null values in 'Invoice':
0
---
Null values in 'InboundDate':
0
---
Null values in 'Supplier':
0
---
Null values in 'OrderDate':
0
---
Null values in 'InvoiceDate':
0
---


In [5]:
# igualmente, no podemos tener valores duplicados en invoice
# # todo correcto porque no hay ningún valor duplicado
df_header["Invoice"].duplicated().sum()

0

In [6]:
# cambiamos el nombre de "supplier" a "ID supplier" para que coincida conel nombre que tiene el csv de suppliers
# luego haremos un merge en base a esta viarble
df_header = df_header.rename(columns={"Supplier" : "IDSupplier"})

In [7]:
# vamos a añadir una columna para el lead time real
# como se describe en el enunciado, obtenemos el lead time restando OrderDate e InboundDate
df_header.dtypes

Invoice        object
InboundDate    object
IDSupplier     object
OrderDate      object
InvoiceDate    object
dtype: object

In [8]:
# pero tal y como están, las variables "InboundDate" y "OrderDate" son strings, por lo que no podremos operar con ellas
df_header["OrderDate"] = pd.to_datetime(df_header["OrderDate"])
df_header["InboundDate"] = pd.to_datetime(df_header["InboundDate"])
# ahora sí podemos crear nuestra nueva columna
df_header["LeadTime"] = df_header["InboundDate"] - df_header["OrderDate"]
# revisamos el resultado final
df_header.head()

,Invoice,InboundDate,IDSupplier,OrderDate,InvoiceDate,LeadTime
0,FFCC141196,2014-05-26,PROV1650,2014-05-10,2014-05-29,16 days
1,FFCC141197,2014-05-20,PROV40000235,2014-04-13,2014-05-23,37 days
2,FFCC141198,2014-05-12,PROV1647,2014-04-27,2014-05-15,15 days
3,FFCC141199,2014-05-19,PROV40001000,2014-03-24,2014-05-22,56 days
4,FFCC141200,2014-05-24,PROV40000850,2014-05-11,2014-05-27,13 days


In [9]:
### DF_INVOICES ###


df_invoices.head()

,Invoice,Quantity,Product,PurchasePrice (Unit),Section
0,FFCC141196,1031,MP04245,0.441840,Seccion A
1,FFCC141197,1931,MP04227,1.680570,Seccion D
2,FFCC141198,360,MP02868,1.159200,Seccion D
3,FFCC141198,240,MP02869,1.214400,Seccion E
4,FFCC141199,18138,MP02700,0.124391,Seccion E


In [10]:
# vamos a empezar mirando si tenemos valores nulos
for column in df_invoices.columns:
    null_values_invoices = df_invoices[column].isnull().sum()
    print(f"Null values in '{column}':")
    print(null_values_invoices)
    print("---")

Null values in 'Invoice':
0
---
Null values in 'Quantity':
0
---
Null values in 'Product':
0
---
Null values in 'PurchasePrice (Unit)':
0
---
Null values in 'Section':
0
---


In [11]:
# solo tenemos precio de unidad y cantidad de producto comprada, vamos a crear una columna con el importe total
# luego poder agrupar por facturas o proveedor
df_invoices["TotalAmount"] = df_invoices["Quantity"] * df_invoices["PurchasePrice (Unit)"]
# redondeamos, para que quede más limpio
df_invoices.TotalAmount = df_invoices.TotalAmount.round(2)

In [12]:
# quitamos la columna de section que no es relevante para este análisis y así dejamos un poco más limpios los datos
df_invoices = df_invoices.drop("Section", axis="columns")

In [13]:
# lo guardamos en un csv para luego trabajar con estos datos en powerbi
df_invoices.to_csv("invoices_processed.csv", index=False, sep = ";")

In [14]:
### DF_SUPPLIER ###


df_supplier.head()

,SupplierName,PaymentMethod,PaymentTerms,IDSupplier,Country,Currency
0,Proveedor 3,RECIBO,1X30,PROV41000270,ES,EUR
1,Proveedor 4,RECIBO,1X60,PROV1344,ES,EUR
2,Proveedor 12,RECIBO,1X85,PROV40000010,ES,EUR
3,Proveedor 13,RECIBO,1X60,PROV40000187,ES,EUR
4,Proveedor 14,RECIBO,1X60,PROV40000200,ES,EUR


In [15]:
# vamos a empezar mirando si tenemos valores nulos
# tenemos valores nulos en "PaymentMethod" y "PaymentTerms"
for column in df_supplier.columns:
    null_values_supplier = df_supplier[column].isnull().sum()
    print(f"Null values in '{column}':")
    print(null_values_supplier)
    print("---")

Null values in 'SupplierName':
0
---
Null values in 'PaymentMethod':
7
---
Null values in 'PaymentTerms':
6
---
Null values in 'IDSupplier':
0
---
Null values in 'Country':
0
---
Null values in 'Currency':
0
---


In [16]:
# miramos los valores únicos, para ver si hay valores que son el mismo, pero se guardan de manera diferente
# por ejemplo, ES y españa
for column in df_supplier.columns:
    unique_values_supplier = df_supplier[column].unique()
    print(f"Unique values in '{column}':")
    print(unique_values_supplier)
    print("---")

Unique values in 'SupplierName':
['Proveedor 3' 'Proveedor 4' 'Proveedor 12' ... 'Proveedor 1192'
 'Proveedor 1195' 'Proveedor 425']
---
Unique values in 'PaymentMethod':
['RECIBO' 'TRANSFE.' 'CONTADO' 'TRANS_FRAC' 'VISA' nan 'GIRO' 'BANCO'
 'CAJA' 'TALON' 'CAN PAGO A' 'CONFIRMING' 'DB' 'CR PAGO A']
---
Unique values in 'PaymentTerms':
['1X30' '1X60' '1X85' '1X45' '1X75' '1X90' '15DIAS' 'ADELANTADO' '8DIAS'
 'REPOSICION' '10DIAS' '1X60 D25' '2X30BL' '3X20AD' 'VISTA' nan '3X30'
 '2X30AD' '2X30%BL' '2X30' '1X45D B/L' '1X0D' 'ADEL/APLA' '2X30A' '30/70'
 '-' '14DIAS' '20DIAS' '2X' '2XADE' '7DIAS' '3X30AD' '2X80' '1X40 B/L'
 '4X30D' '1' '2X85/30' '1X60 B/L' '2 TIMES' '1X0D BL']
---
Unique values in 'IDSupplier':
['PROV41000270' 'PROV1344' 'PROV40000010' ... 'PROV2370' 'PROV2373'
 'PROV1528']
---
Unique values in 'Country':
['ES' 'FR' 'CH' 'españa' 'IN' 'SA' 'DE' 'IE' 'SG' 'JO' 'AE' 'CN' 'PE' 'MX'
 'HU' 'PT' 'BE' 'TR' 'RU' 'KR' 'SE' 'IT' 'NL' 'GB' 'PL' 'GR']
---
Unique values in 'Currency':


In [ ]:
# además de valores nulos, también tenemos "-". 
# Vamos a pasar todos los nulos y los "-" a "NA". Para que haya armonia.
# aunque luego he optado por hacer drop a estas columnas, así que este paso es irrelevante
df_supplier["PaymentMethod"] = df_supplier["PaymentMethod"].fillna("N/A")
df_supplier["PaymentTerms"] = df_supplier["PaymentTerms"].replace([pd.NA, "-"], "N/A")
# se ha detectado que en "Country" existe "españa" y "ES"
# vamos a cambiar el valor "españa" para que esté en sintonía con el resto
df_supplier["Country"] = df_supplier["Country"].replace("españa", "ES")

In [18]:
# tenemos que generar columnas para determinar los lead times teóricos y ver si los clientes lo cumplen
# quitamos a GB por tema Brexit y para Grecia ponemos "GR", aunque el código asignado por EU es "EL", pero no se usa en el dataset
europeCountries = ["AT", "BE", "BG", "CY", "CZ", "DE", "DK", "EE", "GR", "FI", "FR", "GB", "HR", 
                   "HU", "IE", "IT", "LT", "LU" "LV", "MT", "NL", "PL", "PT", "RO", "SE", "SI", "SK"]
conditions_CountryOrigen = [
    (df_supplier["Country"] == "ES"),
     (df_supplier["Country"].isin(europeCountries)),
     (~df_supplier["Country"].isin(europeCountries + ["ES"]))
]
values_CountryOrigen = ["Spanish", "Intracommunity", "Extracommunity"]
df_supplier["CountryOrigen"] = np.select(conditions_CountryOrigen, values_CountryOrigen, default="Unknown")

In [19]:
# ahora que tenemos los países calificados en españoles, intracommunity y extracommunity
# tenemos que determinar su lead time teórico
# proveedores españoles 10 días; proveedores europeos 20 días, proveedores no europeos 45 días
conditions_TheoricalLeadTime = [
    df_supplier["CountryOrigen"] == "Spanish",
    df_supplier["CountryOrigen"] == "Intracommunity",
    df_supplier["CountryOrigen"] == "Extracommunity"
]
values_TheoricalLeadTime = ["10 days", "20 days", "45 days"]
df_supplier["TheoricalLeadTime"] = np.select(conditions_TheoricalLeadTime, values_TheoricalLeadTime, default="Unknown")

In [20]:
# nos quedamos solo con las columnas relevantes y dejamos atrás "PaymentMethod" y "PaymentTerm"
# currency la mantengo porque ahora solo paga en EUR y USD, que tienen un valor similar y se puede hacer el cambio fácil
# pero si fuéramos a ser pagados en otras monedas con valores más dispares, entiendo que sería interesante usar un conversor de moneda
# se podría hacer uso, por ejemplo, del proyecto CurrencyConverter
df_supplier = df_supplier.drop(["PaymentMethod", "PaymentTerms"], axis="columns")

In [21]:
### DF_PRODUCT ###


df_product.head()

,Product,Type,Division,Group,ShortDescription,Description
0,NaN,Tipo 11178,División 3,Grupo 2,Texto Search Descripción 12,Texto Descripción 15
1,DE0001,Tipo 3,División 3,Grupo 2,Texto Search Descripción 8,Texto Descripción 12
2,DE0002,Tipo 3,División 3,Grupo 2,Texto Search Descripción 8,Texto Descripción 12
3,DE0003,Tipo 3,División 3,Grupo 2,Texto Search Descripción 8,Texto Descripción 12
4,DE0004,Tipo 3,División 3,Grupo 2,Texto Search Descripción 8,Texto Descripción 12


In [22]:
# vamos a empezar mirando si tenemos valores nulos
# tenemos valores nulos en "Product", "Type", "Division", "Group", "ShortDescription"
for column in df_product.columns:
    null_values_product = df_product[column].isnull().sum()
    print(f"Null values in '{column}':")
    print(null_values_product)
    print("---")

Null values in 'Product':
1
---
Null values in 'Type':
3395
---
Null values in 'Division':
23
---
Null values in 'Group':
21
---
Null values in 'ShortDescription':
103
---
Null values in 'Description':
0
---


In [23]:
# vamos a ver los duplicados de "Product", que es nuestra key
duplicados = df_product[df_product.duplicated(subset=["Product"], keep=False)]
print(duplicados)


       Product Type    Division    Group             ShortDescription  \
286  DEP024201  NaN  División 3  Grupo 2  Texto Search Descripción 33   
287  DEP024201  NaN         NaN      NaN  Texto Search Descripción 33   
288  DEP024201  NaN         NaN      NaN  Texto Search Descripción 33   

              Description  
286  Texto Descripción 25  
287  Texto Descripción 25  
288  Texto Descripción 25  


In [24]:
# vamos a quitar esas dos líneas que parecen haber sido un error del corta pega
# igualmente, entiendo que habría que contrastas con el cliente que no se ha quedado ningún producto atrás
df_product = df_product.drop([287, 288])

In [25]:
# el null en "Product" nos puede dar problemas, porque es la key que lo une con la factura
# se podría preguntar al cliente, para esta práctica, voy a borrarlo
df_product = df_product.dropna(subset=["Product"])
# el resto, los vamos a sustituir por "N/A"
df_product[["Type", "Division", "Group", "ShortDescription"]] = df_product[["Type", "Division", "Group", "ShortDescription"]].fillna( "N/A")

In [26]:
# pensándolo mejor, vamos a quedarnos solo con la ID del producto y short description
# así podremos identificar qué productos son los que se facturan y demás
# porque entiendo que el resto de columnas nos es indiferente para el análisis
df_product = df_product.drop(["Type", "Division", "Group", "Description"], axis="columns")

In [27]:
df_product.to_csv("product_processed.csv", index=False, sep = ";")

In [28]:
# vamos a juntar invoice header con supplier
# este merge lo hago porque es la manera que se me ha ocurrido para comprobar si nuestros proveedores están cumpliendo con los leads teóricos
# comprobaremos el lead time real con la columna de lead time teórico que hemos diseñado antes
# aunque esta forma de trabajar supongo que hace más difícil actualizar la dimensión de suppliers, que ha sido absorbida por el hecho
# quizás hubiera sido mejor dejarlo como está y hacer este cálculo directamente en powerbi
df = pd.merge(df_header, df_supplier, on = "IDSupplier", how = "left")
df.head()

,Invoice,InboundDate,IDSupplier,OrderDate,InvoiceDate,LeadTime,SupplierName,Country,Currency,CountryOrigen,TheoricalLeadTime
0,FFCC141196,2014-05-26,PROV1650,2014-05-10,2014-05-29,16 days,Proveedor 539,ES,EUR,Spanish,10 days
1,FFCC141197,2014-05-20,PROV40000235,2014-04-13,2014-05-23,37 days,Proveedor 15,ES,EUR,Spanish,10 days
2,FFCC141198,2014-05-12,PROV1647,2014-04-27,2014-05-15,15 days,Proveedor 536,ES,EUR,Spanish,10 days
3,FFCC141199,2014-05-19,PROV40001000,2014-03-24,2014-05-22,56 days,Proveedor 24,FR,EUR,Intracommunity,20 days
4,FFCC141200,2014-05-24,PROV40000850,2014-05-11,2014-05-27,13 days,Proveedor 21,ES,EUR,Spanish,10 days


In [29]:
# vamos a definir una nueva columna para ver si nuestros proveedores están cumpliendo con su lead time
# una columna booleana de True si han entragado por debajo del lead time teórico
# y False si han entregado por encima del lead time teórico
df["GoodLeadTime"] = df["LeadTime"] <= df["TheoricalLeadTime"]

In [30]:
df.to_csv("invoice_supplier_processed.csv", index=False, sep = ";")

In [31]:
# en este ETL se han limpiado los df que teníamos, analizando valores nulos, duplicados, erróneos, normalizar datos, correción de formatos...
# también se han creado columnas nuevas previo a la creación del dashboard en powerBI
# el resultado final son tres archivos csv
# header + suppliers: donde tenemos toda la información referente a los lead times del proveedor
# -> invoice, inbound date, id suppiler, order date, invoice date, leadtime, supplier name, country, currency, country origen, theorical lead time
# invoice_processed: donde hemos calculado el valor total de cada línea de producto comprada en cada factura
# -> invoice, quantity, product, purchase price, total amount
# product_processed: donde solo hemos limpiado un poco y dejado atrás columnas que no nos servían
# -> product, short descriptio.
# con esto, nos podemos ir a powerBI